In [0]:
from pyspark.sql import functions as F

bronze_path = "/Volumes/sales_lakehouse/bronze/bronze_volume/Targets.csv"

df = (spark.read
      .option("header", True)
      .option("delimiter", "\t")
      .csv(bronze_path))
display(df)
df.printSchema()

## Rename to snake_case

In [0]:
df = (df.withColumnRenamed("EmployeeID", "employee_id")
        .withColumnRenamed("Target", "target")
        .withColumnRenamed("TargetMonth", "target_month"))

## Cast Employee_ID

In [0]:
df = df.withColumn("employee_id", F.col("employee_id").cast("int"))

## Parse target_month (format: Friday, December 1, 2017)

In [0]:
from pyspark.sql import functions as F


df = df.withColumn(
    "target_month",
    F.to_date(
        F.regexp_replace(F.col("target_month"), r"^[A-Za-z]+,\s*", ""),
        "MMMM d, yyyy"
    )
)

## Clean currency: "$500,000" -> _500000_

In [0]:
df = df.withColumn("target", F.regexp_replace(F.col("target"), r"[\$,]", "").cast("double"))

In [0]:
(df.write
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .format("delta")
 .saveAsTable("sales_lakehouse.silver.salesperson_region"))


In [0]:
display(spark.table("sales_lakehouse.silver.salesperson_region"))
spark.table("sales_lakehouse.silver.targets").printSchema()